In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.model_selection import StratifiedKFold

CURRENT_DIR = Path.cwd().resolve()

PROJECT_ROOT = (
    CURRENT_DIR.parent
    if CURRENT_DIR.name == "notebooks"
    else CURRENT_DIR
)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.feature_engineering import (
    prepare_features,
    add_title_hierarchy_features,
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports"

TARGET_COLUMN = "Цена"

train = pd.read_parquet(
    PROCESSED_DIR / "train_canonical.parquet"
)

y = train[TARGET_COLUMN].copy()

X_v5 = add_title_hierarchy_features(
    prepare_features(
        train.drop(columns=[TARGET_COLUMN])
    )
)

RAW_DUPLICATE_NUMERIC_COLUMNS = [
    "Пробег",
    "Расход",
    "Количество цилиндров",
    "Двери",
    "Количество кресел",
]

EXCLUDED_COLUMNS = [
    "car_id",
    "Предложение",
] + RAW_DUPLICATE_NUMERIC_COLUMNS

# V5: raw «Полное название» намеренно остаётся.
feature_columns_v5 = [
    column
    for column in X_v5.columns
    if column not in EXCLUDED_COLUMNS
]

X_model_v5 = X_v5[
    feature_columns_v5
].copy()

numeric_columns_v5 = X_model_v5.select_dtypes(
    include=["number", "bool"]
).columns.tolist()

categorical_columns_v5 = [
    column
    for column in feature_columns_v5
    if column not in numeric_columns_v5
]

for column in categorical_columns_v5:
    X_model_v5[column] = (
        X_model_v5[column]
        .fillna("__MISSING__")
        .astype(str)
    )

assert "Полное название" in X_model_v5.columns
assert X_model_v5.shape[1] == 59

cv_target_bins = pd.qcut(
    y,
    q=10,
    labels=False,
    duplicates="drop",
).to_numpy()

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

print("X_model_v5:", X_model_v5.shape)
print("Numeric:", len(numeric_columns_v5))
print("Categorical:", len(categorical_columns_v5))

X_model_v5: (8340, 59)
Numeric: 40
Categorical: 19


In [2]:
def mape_percent(y_true, y_pred) -> float:
    return mean_absolute_percentage_error(y_true, y_pred) * 100


oof_v5_predictions = np.zeros(len(y))
v5_fold_results = []

for fold, (train_fold_idx, valid_fold_idx) in enumerate(
    cv.split(X_model_v5, cv_target_bins),
    start=1,
):
    print(f"\n{'=' * 60}")
    print(f"CatBoost v5 fold {fold}/5")
    print(f"{'=' * 60}")

    X_train_fold = X_model_v5.iloc[train_fold_idx].copy()
    X_valid_fold = X_model_v5.iloc[valid_fold_idx].copy()

    y_train_fold = y.iloc[train_fold_idx].copy()
    y_valid_fold = y.iloc[valid_fold_idx].copy()

    model_fold = CatBoostRegressor(
        loss_function="RMSE",
        iterations=3000,
        learning_rate=0.05,
        depth=8,
        l2_leaf_reg=5,
        random_seed=42,
        verbose=500,
        allow_writing_files=False,
    )

    model_fold.fit(
        X_train_fold,
        np.log1p(y_train_fold),
        cat_features=categorical_columns_v5,
        eval_set=(
            X_valid_fold,
            np.log1p(y_valid_fold),
        ),
        use_best_model=True,
        early_stopping_rounds=200,
    )

    fold_predictions = np.maximum(
        np.expm1(
            model_fold.predict(X_valid_fold)
        ),
        1,
    )

    oof_v5_predictions[valid_fold_idx] = fold_predictions

    fold_mape = mape_percent(
        y_valid_fold,
        fold_predictions,
    )

    v5_fold_results.append(
        {
            "fold": fold,
            "best_iteration": model_fold.get_best_iteration(),
            "validation_mape_pct": fold_mape,
        }
    )

    print(
        f"Fold {fold} MAPE: {fold_mape:.3f}% | "
        f"best iteration: {model_fold.get_best_iteration()}"
    )


CatBoost v5 fold 1/5
0:	learn: 0.6544254	test: 0.6474238	best: 0.6474238 (0)	total: 227ms	remaining: 11m 20s
500:	learn: 0.1493383	test: 0.2104536	best: 0.2104536 (500)	total: 41.9s	remaining: 3m 29s
1000:	learn: 0.1115406	test: 0.2029048	best: 0.2028414 (980)	total: 1m 18s	remaining: 2m 37s
1500:	learn: 0.0906444	test: 0.2006227	best: 0.2006227 (1500)	total: 1m 58s	remaining: 1m 58s
2000:	learn: 0.0747698	test: 0.1996945	best: 0.1996945 (2000)	total: 2m 38s	remaining: 1m 19s
2500:	learn: 0.0627711	test: 0.1989965	best: 0.1989965 (2500)	total: 3m 17s	remaining: 39.4s
2999:	learn: 0.0530521	test: 0.1984778	best: 0.1984689 (2900)	total: 4m 5s	remaining: 0us

bestTest = 0.1984688888
bestIteration = 2900

Shrink model to first 2901 iterations.
Fold 1 MAPE: 13.408% | best iteration: 2900

CatBoost v5 fold 2/5
0:	learn: 0.6507823	test: 0.6568553	best: 0.6568553 (0)	total: 121ms	remaining: 6m 4s
500:	learn: 0.1574818	test: 0.1925524	best: 0.1925524 (500)	total: 56.9s	remaining: 4m 43s
1000:	

In [3]:
v5_fold_results_df = pd.DataFrame(v5_fold_results)

print(
    "OOF MAPE v5:",
    f"{mape_percent(y, oof_v5_predictions):.3f}%"
)

display(v5_fold_results_df)

v5_oof_report = pd.DataFrame(
    {
        "car_id": train["car_id"].to_numpy(),
        "y_true": y.to_numpy(),
        "title_v5_pred": oof_v5_predictions,
    }
)

v5_oof_report.to_parquet(
    REPORTS_DIR / "catboost_title_hierarchy_v5_oof_clean.parquet",
    index=False,
)

OOF MAPE v5: 12.755%


,fold,best_iteration,validation_mape_pct
0,1,2900,13.407819
1,2,2987,12.322104
2,3,2974,12.732862
3,4,2941,12.987029
4,5,2955,12.326795
